# RF-DETR Small — detector de código de barras (substituto do Roboflow)

Treina um detector **próprio e gratuito** para ocupar o lugar de `roboflow_barcode_candidates()` no `barcode_pipeline.py`.
O detector só precisa achar **onde** estão os códigos; a leitura continua com zxing/pyzbar/OpenCV + camadas.

**Licença** — usamos apenas componentes Apache 2.0 (uso comercial liberado, sem obrigação de abrir código):
- pacote `rfdetr` **sem** o extra `plus`;
- modelo **RFDETRSmall**. Não usar Atto/Femto/Pico/XL/2XL nem `rfdetr_plus` (licença PML 1.0).

**Antes de rodar**
1. `Ambiente de execução → Alterar tipo → GPU (T4 serve)`.
2. Ter o dataset no Google Drive (seção 3) em formato **COCO** ou **YOLO**.
3. **Não** colocar as 71 imagens de `imagens/imagens` no treino — elas são o teste que compara com o Roboflow (`avaliacao_roboflow_v2.csv`).

## 1. GPU e instalação

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Versão fixada para o notebook continuar reprodutível quando sair release nova.
!pip install -q "rfdetr[train,augment,loggers,onnx]==1.10.1"

In [ ]:
import rfdetr, torch
print("rfdetr", rfdetr.__version__ if hasattr(rfdetr, "__version__") else "", "| torch", torch.__version__,
      "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Sem GPU: troque o tipo de ambiente de execução para GPU."

## 2. Google Drive e pastas

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_DIR   = Path("/content/drive/MyDrive/barcode_rfdetr")   # tudo que precisa sobreviver fica aqui
RUN_NAME    = "rfdetr_small_v2"                               # v1 = só dataset de produto; v2 = + etiquetas de ONU
OUTPUT_DIR  = DRIVE_DIR / "runs" / RUN_NAME                   # checkpoints (no Drive: sobrevive a desconexão)
# Ponto de partida do treino: o melhor modelo da rodada anterior (fine-tuning).
# Se não existir, a seção 4.1 começa dos pesos padrão do RF-DETR Small.
PREV_WEIGHTS = DRIVE_DIR / "runs" / "rfdetr_small_v1" / "checkpoint_best_total.pth"
DATASET_DIR = Path("/content/dataset")                        # cópia local: treino lê muito mais rápido que do Drive
for p in (DRIVE_DIR, OUTPUT_DIR):
    p.mkdir(parents=True, exist_ok=True)
print(DRIVE_DIR, OUTPUT_DIR, sep="\n")

## 3. Dataset

Escolha **uma** das opções. O resultado final precisa ser `DATASET_DIR` em um destes formatos (o `rfdetr` detecta sozinho):

```
COCO                                   YOLO
dataset/                               dataset/
  train/  imagens + _annotations.coco.json   data.yaml
  valid/  imagens + _annotations.coco.json   train/images  train/labels
  test/   imagens + _annotations.coco.json   valid/images  valid/labels
                                             test/images   test/labels   (test é opcional)
```

### 3A. Zip que já está no Drive (COCO ou YOLO)

In [ ]:
DATASET_ZIP = DRIVE_DIR / "dataset.zip"   # <-- ajuste o nome

import shutil, zipfile
if DATASET_ZIP.exists():
    shutil.rmtree(DATASET_DIR, ignore_errors=True)
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(DATASET_DIR)
    # Se o zip tiver uma pasta-raiz única (ex.: dataset/meu_dataset/train), sobe um nível.
    inner = [p for p in DATASET_DIR.iterdir() if p.is_dir()]
    if len(inner) == 1 and not (DATASET_DIR / "train").exists():
        DATASET_DIR = inner[0]
    print("Dataset em", DATASET_DIR, "->", sorted(p.name for p in DATASET_DIR.iterdir()))
else:
    print("Zip não encontrado, use 3B ou 3C.")

### 3B. Dataset público do Roboflow Universe (download gratuito)

Baixar um dataset **público** do Universe é gratuito com uma conta free (a API key fica em *Settings → API Keys*).
Confira a licença na página do dataset (normalmente CC BY 4.0, que permite uso comercial com atribuição).
Preencha `WORKSPACE`, `PROJECT` e `VERSION` com os valores da URL do dataset: `universe.roboflow.com/<WORKSPACE>/<PROJECT>/dataset/<VERSION>`.

In [ ]:
# Fork do dataset público "barcode-detection-mziov" (CC BY 4.0: uso comercial liberado, exige crédito à fonte).
# Projeto: https://app.roboflow.com/mariana-silva/barcode-detection-mziov-dem6u/1
WORKSPACE, PROJECT, VERSION = "mariana-silva", "barcode-detection-mziov-dem6u", 1

if WORKSPACE:
    from getpass import getpass
    from roboflow import Roboflow
    rf = Roboflow(api_key=getpass("Roboflow API key: "))
    ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("coco", location=str(DATASET_DIR))
    print("Dataset em", DATASET_DIR, "->", sorted(p.name for p in DATASET_DIR.iterdir()))

### 3C. Dataset do YOLOv4 (Darknet) que vocês já rotularam

O Darknet guarda cada imagem com um `.txt` ao lado e listas `train.txt` / `valid.txt`. Esta célula converte para o
formato YOLO de pastas. Os rótulos (`classe cx cy w h`, normalizados) são os mesmos, então nada precisa ser re-rotulado.
Suba para o Drive um zip com as imagens + `.txt`, o `obj.names` e as listas `train.txt`/`valid.txt`.

In [ ]:
DARKNET_ZIP = DRIVE_DIR / "darknet_dataset.zip"   # <-- ajuste o nome

import random
if DARKNET_ZIP.exists():
    raw = Path("/content/darknet_raw")
    shutil.rmtree(raw, ignore_errors=True)
    with zipfile.ZipFile(DARKNET_ZIP) as z:
        z.extractall(raw)

    names_file = next(raw.rglob("obj.names"), None)
    assert names_file, "obj.names não encontrado no zip do Darknet."
    class_names = [l.strip() for l in names_file.read_text().splitlines() if l.strip()]
    images = {p.name: p for p in raw.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}}

    def listed(fname):
        f = next(raw.rglob(fname), None)
        return [Path(l.strip()).name for l in f.read_text().splitlines() if l.strip()] if f else None

    train_names, valid_names = listed("train.txt"), listed("valid.txt") or listed("test.txt")
    if not train_names or not valid_names:   # sem listas: divide 85/15
        allnames = sorted(images); random.Random(0).shuffle(allnames)
        cut = int(len(allnames) * 0.85)
        train_names, valid_names = allnames[:cut], allnames[cut:]

    DATASET_DIR = Path("/content/dataset")
    shutil.rmtree(DATASET_DIR, ignore_errors=True)
    for split, names in (("train", train_names), ("valid", valid_names)):
        (DATASET_DIR / split / "images").mkdir(parents=True)
        (DATASET_DIR / split / "labels").mkdir(parents=True)
        for name in names:
            img = images.get(name)
            if img is None:
                continue
            shutil.copy(img, DATASET_DIR / split / "images" / name)
            lbl = img.with_suffix(".txt")
            dst = DATASET_DIR / split / "labels" / (img.stem + ".txt")
            dst.write_text(lbl.read_text() if lbl.exists() else "")   # sem .txt = imagem negativa
    yaml_lines = ["train: train/images", "val: valid/images", f"nc: {len(class_names)}", f"names: {class_names}"]
    (DATASET_DIR / "data.yaml").write_text("\n".join(yaml_lines) + "\n")
    print("Classes:", class_names, "| train:", len(train_names), "| valid:", len(valid_names))

### 3C+. Juntar datasets extras (treino v2)

Junta ao dataset principal (3B) os pacotes COCO que estiverem no Drive em `barcode_rfdetr/extras/`:
- `sinteticos_onu.zip` — etiquetas sintéticas de ONU (gerador `rfdetr/gerar_etiquetas_sinteticas.py`);
- o zip exportado do Roboflow com as fotos do Google **já revisadas** (formato COCO).

As imagens extras entram nos mesmos splits do pacote (`train`/`valid`/`test`); pacote sem splits vai todo para `train`.
**Nunca coloque aqui as 71 fotos de avaliação.**

In [ ]:
import json, shutil, zipfile
EXTRAS_DIR = DRIVE_DIR / "extras"
EXTRAS_DIR.mkdir(exist_ok=True)

def base_category(coco):
    # categoria com anotações (o export do Roboflow tem também uma "supercategoria" vazia)
    usadas = {a["category_id"] for a in coco["annotations"]}
    cand = [c for c in coco["categories"] if c["id"] in usadas] or coco["categories"]
    return cand[-1]["id"]

def merge_coco_split(src_dir, dst_dir, prefix):
    src_ann = src_dir / "_annotations.coco.json"
    if not src_ann.exists():
        return 0, 0
    src = json.loads(src_ann.read_text())
    dst_ann = dst_dir / "_annotations.coco.json"
    dst = json.loads(dst_ann.read_text())
    cat = base_category(dst)
    img_off = max([i["id"] for i in dst["images"]], default=0) + 1
    ann_off = max([a["id"] for a in dst["annotations"]], default=0) + 1
    novo_id = {}
    for k, im in enumerate(src["images"]):
        nome = f"{prefix}_{Path(im['file_name']).name}"
        shutil.copy(src_dir / im["file_name"], dst_dir / nome)
        novo_id[im["id"]] = img_off + k
        dst["images"].append({**im, "id": img_off + k, "file_name": nome})
    n = 0
    for a in src["annotations"]:
        if a["image_id"] in novo_id:
            dst["annotations"].append({**a, "id": ann_off + n, "image_id": novo_id[a["image_id"]], "category_id": cat})
            n += 1
    dst_ann.write_text(json.dumps(dst))
    return len(novo_id), n

zips = sorted(EXTRAS_DIR.glob("*.zip"))
print("Pacotes extras:", [z.name for z in zips] or "nenhum (coloque os zips em " + str(EXTRAS_DIR) + ")")
for z in zips:
    tmp = Path("/content/extra") / z.stem
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(z) as f:
        f.extractall(tmp)
    raiz = next((p.parent for p in tmp.rglob("_annotations.coco.json")), None)
    if raiz is None:
        print("  ", z.name, "sem _annotations.coco.json, ignorado"); continue
    raiz = raiz.parent if raiz.name in ("train", "valid", "test") else raiz
    splits = [d for d in ("train", "valid", "test") if (raiz / d / "_annotations.coco.json").exists()]
    pares = [(raiz / d, d) for d in splits] or [(raiz, "train")]
    for origem, destino in pares:
        if not (DATASET_DIR / destino / "_annotations.coco.json").exists():
            destino = "train"
        ni, na = merge_coco_split(origem, DATASET_DIR / destino, z.stem)
        print(f"   {z.name}: {origem.name} -> {destino}: +{ni} imagens, +{na} caixas")

### 3D. Conferência do dataset

Conta imagens e caixas por split e verifica se alguma das 71 imagens de avaliação vazou para o treino.

In [ ]:
import json, yaml
from collections import Counter

EVAL_NAMES = set()   # preenchido abaixo se avaliacao_roboflow_v2.csv estiver no DRIVE_DIR
eval_csv = DRIVE_DIR / "avaliacao_roboflow_v2.csv"
if eval_csv.exists():
    import pandas as pd
    EVAL_NAMES = set(pd.read_csv(eval_csv, encoding="utf-8-sig")["arquivo"].map(lambda s: Path(s).stem))

def summarize(dataset_dir):
    dataset_dir = Path(dataset_dir)
    report = {}
    if (dataset_dir / "data.yaml").exists() or (dataset_dir / "data.yml").exists():
        y = yaml.safe_load(next(dataset_dir.glob("data.y*ml")).read_text())
        print("Formato YOLO | classes:", y.get("names"))
        for split in ("train", "valid", "val", "test"):
            img_dir = dataset_dir / split / "images"
            if not img_dir.exists():
                continue
            imgs = [p for p in img_dir.iterdir() if p.is_file()]
            boxes, classes = 0, Counter()
            for lbl in (dataset_dir / split / "labels").glob("*.txt"):
                for line in lbl.read_text().splitlines():
                    if line.strip():
                        boxes += 1; classes[line.split()[0]] += 1
            report[split] = (imgs, boxes, classes)
    else:
        print("Formato COCO")
        for split in ("train", "valid", "test"):
            ann = dataset_dir / split / "_annotations.coco.json"
            if not ann.exists():
                continue
            data = json.loads(ann.read_text())
            cats = {c["id"]: c["name"] for c in data["categories"]}
            classes = Counter(cats[a["category_id"]] for a in data["annotations"])
            imgs = [Path(i["file_name"]) for i in data["images"]]
            report[split] = (imgs, len(data["annotations"]), classes)
    for split, (imgs, boxes, classes) in report.items():
        leaked = {Path(p).stem for p in imgs} & EVAL_NAMES
        print(f"  {split:5s}: {len(imgs):5d} imagens | {boxes:6d} caixas | por classe: {dict(classes)}"
              + (f" | ATENÇÃO: {len(leaked)} imagens de avaliação aqui!" if leaked and split != 'test' else ""))
    return report

_ = summarize(DATASET_DIR)

import supervision as sv

def load_split(dataset_dir, split):
    dataset_dir = Path(dataset_dir)
    if (dataset_dir / split / "_annotations.coco.json").exists():
        return sv.DetectionDataset.from_coco(str(dataset_dir / split), str(dataset_dir / split / "_annotations.coco.json"))
    return sv.DetectionDataset.from_yolo(str(dataset_dir / split / "images"), str(dataset_dir / split / "labels"),
                                         str(next(dataset_dir.glob("data.y*ml"))))

### 3E. Olhar algumas imagens com as caixas

Rótulo ruim = modelo ruim. Vale gastar 2 minutos aqui: as caixas devem cobrir **as barras inteiras**, sem sobrar muito.

In [ ]:
import matplotlib.pyplot as plt, random

train_ds = load_split(DATASET_DIR, "train")
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, idx in zip(axes.flat, random.Random(1).sample(range(len(train_ds)), min(6, len(train_ds)))):
    path, image, ann = train_ds[idx]
    ax.imshow(sv.BoxAnnotator(thickness=4).annotate(image.copy(), ann)[..., ::-1])
    ax.set_title(f"{Path(path).name} — {len(ann)} caixas", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Treino

Escolhas pensadas para código de barras:
- **`resolution=640`** (padrão do Small é 512). Barras são finas; mais resolução ajuda a achar código pequeno.
  No Small precisa ser múltiplo de 32 — se faltar memória, volte para 576 ou 512.
- **Augmentations** moderadas: rotação pequena, perspectiva, blur, ruído e brilho/contraste — o que acontece com foto
  de celular em campo. Rotação grande infla demais as caixas (elas são retas, não giradas).
- **Checkpoints no Drive** a cada 5 épocas: se o Colab cair, a seção 5 retoma de onde parou.
- `batch_size=4` × `grad_accum_steps=4` = lote efetivo 16, recomendado pela documentação para T4.

In [ ]:
from rfdetr import RFDETRSmall

AUG_BARCODE = {
    "HorizontalFlip": {"p": 0.5},
    "Affine": {"rotate": (-10, 10), "scale": (0.8, 1.2), "p": 0.5},
    "Perspective": {"scale": (0.02, 0.06), "p": 0.3},
    "RandomBrightnessContrast": {"brightness_limit": 0.25, "contrast_limit": 0.25, "p": 0.5},
    "GaussianBlur": {"blur_limit": (3, 5), "p": 0.2},
    "GaussNoise": {"p": 0.2},
    "CLAHE": {"p": 0.1},
}

TRAIN_ARGS = dict(
    dataset_dir=str(DATASET_DIR),
    output_dir=str(OUTPUT_DIR),
    resolution=640,
    epochs=10,            # v2 parte do modelo v1 (fine-tuning): menos épocas bastam
    batch_size=4,
    grad_accum_steps=4,
    lr=1e-4,
    amp_dtype="fp16",     # T4 não tem bfloat16 nativo: fp16 deixou a época ~2,5x mais rápida (18 -> 7 min)
    early_stopping=True,
    early_stopping_patience=15,
    checkpoint_interval=5,
    aug_config=AUG_BARCODE,
    tensorboard=True,
)
print("Configuração pronta. Treino do zero: rode a célula abaixo. Retomar: pule para a seção 5.")

### 4.1 Treino novo

**Não rode esta célula para retomar** — ela começa da época 1. Para continuar um treino interrompido, use a seção 5.
Parte de `PREV_WEIGHTS` (modelo da rodada anterior) quando ele existe.

In [ ]:
if PREV_WEIGHTS.exists():
    print("Fine-tuning a partir de", PREV_WEIGHTS)
    model = RFDETRSmall(pretrain_weights=str(PREV_WEIGHTS), num_classes=1)
else:
    print("PREV_WEIGHTS não encontrado: começando dos pesos padrão do RF-DETR Small")
    model = RFDETRSmall()
model.train(**TRAIN_ARGS)

## 5. Retomar treino (só se o Colab desconectar)

Rode as seções 1, 2, 3B, 3C+ e 3D de novo (o Colab apaga o dataset local), depois a célula de configuração da seção 4
(sem a 4.1) e esta célula. O log deve continuar da época em que parou, não da época 1.

In [ ]:
# rfdetr 1.10 salva last.ckpt a cada época e checkpoint_<época>.ckpt a cada checkpoint_interval.
last = OUTPUT_DIR / "last.ckpt"
if not last.exists():
    periodic = sorted(OUTPUT_DIR.glob("checkpoint_*.ckpt"), key=lambda p: int(p.stem.split("_")[1]))
    assert periodic, f"Nenhum checkpoint .ckpt em {OUTPUT_DIR}"
    last = periodic[-1]
print("Retomando de", last.name)
model = RFDETRSmall()
model.train(**TRAIN_ARGS, resume=str(last))

## 6. Curvas de treino

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{OUTPUT_DIR}"

## 7. Avaliação no split de teste — recall e precisão das caixas

Para o pipeline o número que importa é o **recall**: caixa que o detector não acha é código que não é lido.
Caixa sobrando custa só tempo (o decodificador descarta). Por isso o pipeline usa confiança baixa (0.10);
aqui medimos em vários limiares para escolher o valor.

In [ ]:
import numpy as np

# best_total só é gerado quando o treino termina; se foi interrompido, use o best_ema.
BEST = OUTPUT_DIR / "checkpoint_best_total.pth"
if not BEST.exists():
    BEST = OUTPUT_DIR / "checkpoint_best_ema.pth"
print("Avaliando", BEST.name)
best_model = RFDETRSmall(pretrain_weights=str(BEST), resolution=640, num_classes=1)

def box_iou(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    tl = np.maximum(a[:, None, :2], b[None, :, :2]); br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.prod(np.clip(br - tl, 0, None), axis=2)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def evaluate(model, ds, thresholds=(0.1, 0.2, 0.3, 0.5), iou_min=0.5):
    preds = []
    for path, image, gt in ds:
        det = model.predict(image[..., ::-1].copy(), threshold=min(thresholds))   # BGR -> RGB
        preds.append((path, gt, det))
    for t in thresholds:
        tp = fp = fn = 0
        for _, gt, det in preds:
            keep = det.confidence >= t
            p = det.xyxy[keep]
            iou = box_iou(gt.xyxy, p)
            matched_gt, matched_p = set(), set()
            for i, j in sorted(zip(*np.nonzero(iou >= iou_min)), key=lambda ij: -iou[ij]):
                if i not in matched_gt and j not in matched_p:
                    matched_gt.add(i); matched_p.add(j)
            tp += len(matched_gt); fn += len(gt) - len(matched_gt); fp += len(p) - len(matched_p)
        print(f"conf>={t:.2f}  recall={tp / max(tp + fn, 1):.3f}  precisão={tp / max(tp + fp, 1):.3f}  "
              f"(TP={tp} FN={fn} FP={fp})")
    return preds

split = "test" if (DATASET_DIR / "test").exists() else "valid"
print("Split:", split)
preds = evaluate(best_model, load_split(DATASET_DIR, split))

## 8. Teste visual nas fotos reais de etiqueta

Suba um zip com as 71 imagens de `imagens/imagens` para o Drive. Elas **não** estão no treino, então é o teste honesto.
A comparação final (códigos lidos vs. Roboflow) é feita no PC, com o pipeline completo — seção 10.

In [ ]:
REAL_ZIP = DRIVE_DIR / "imagens_avaliacao.zip"   # <-- ajuste o nome
REAL_DIR = Path("/content/imagens_avaliacao")

if REAL_ZIP.exists():
    shutil.rmtree(REAL_DIR, ignore_errors=True)
    with zipfile.ZipFile(REAL_ZIP) as z:
        z.extractall(REAL_DIR)
    import cv2
    real = sorted(p for p in REAL_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    print(len(real), "imagens")
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    for ax, p in zip(axes.flat, real[:12]):
        bgr = cv2.imread(str(p))
        det = best_model.predict(bgr[..., ::-1].copy(), threshold=0.1)
        out = sv.BoxAnnotator(thickness=6).annotate(bgr.copy(), det)
        out = sv.LabelAnnotator(text_scale=1.2).annotate(out, det, [f"{c:.2f}" for c in det.confidence])
        ax.imshow(out[..., ::-1]); ax.set_title(f"{p.name} — {len(det)} caixas", fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()

## 9. Salvar o modelo

`checkpoint_best_total.pth` já está no Drive (`OUTPUT_DIR`) — é o arquivo que vai para o servidor.
O ONNX é opcional: serve para rodar com `onnxruntime` sem instalar PyTorch no servidor.

In [ ]:
print("Pesos para o pipeline:", BEST, f"({BEST.stat().st_size / 1e6:.1f} MB)")

EXPORT_ONNX = False
if EXPORT_ONNX:
    onnx_path = best_model.export(output_dir=str(OUTPUT_DIR / "onnx"))
    print("ONNX:", onnx_path)

## 10. Próximo passo (no PC): plugar no pipeline e comparar com o Roboflow

1. Baixar `checkpoint_best_total.pth` do Drive para `Barcode-segmentation/models/`.
2. `pip install "rfdetr==1.10.1"` no venv do projeto (para inferência não precisa do extra `train`).
3. Trocar o detector em `roboflow_barcode_candidates()` pelo RF-DETR (mesma saída: caixas + confiança).
4. Rodar a avaliação nas 71 imagens e comparar com `avaliacao_roboflow_v2.csv`:
   meta = **≥ 181 códigos lidos** e as mesmas 68 imagens com leitura.